# 03 — Baseline Models

> ## ⚠ Scope: participant001 only
>
> `split_manifest.json` lists one participant, and `02_dataset_assembly.ipynb` still raises
> `NotImplementedError` on subject folds. **Every number in this notebook is n = 1.** Do not quote
> any of it as a dataset-level result.
>
> **Re-run on participants 001–004 once notebook 02 derives `max_disp` against a per-session
> resting reference.** A low session correlation does not mean a failed wearing — `max_disp`
> follows whichever marker sits furthest from its stored baseline, which need not respond to grip.
> Re-referencing recovers p003 s9 from r 0.394 to 0.892 and p004 s10 from 0.248 to 0.715.
>
> Refer to `docs/participants/cross-participant-analysis.md` §6, and run
> `python tools/coupling_check.py recordings/<participant_dir> --shift-sweep` before judging any
> session.
>
> **No session is excluded** on coupling grounds. Mount-to-mount variation is what SO4 measures.


## Section 0 — Setup

Fail loudly at the top instead of quietly at cell 40. Three jobs: (a) prove which interpreter is executing, (b) make core/ importable regardless of where the kernel's cwd landed, (c) fix the seed and the MLflow experiment name once, so nothing downstream invents its own.

In [ ]:
import pandas as pd
import numpy as np
import scipy
from pathlib import Path
import sys 
import json

import sklearn
from sklearn.linear_model import Ridge
import xgboost
import optuna
import mlflow
import torch
import torch.nn as nn

print(sys.executable)
print(f'sklearn: {sklearn.__version__}\nxgboost: {xgboost.__version__}\noptuna: {optuna.__version__}\
    \nmlflow: {mlflow.__version__}\ntorch: {torch.__version__}\npandas: {pd.__version__}\nnumpy: {np.__version__}\nscipy: {scipy.__version__}')

CURRENT_FOLDER = Path.cwd()
ROOT_FOLDER = CURRENT_FOLDER.parent
assert (ROOT_FOLDER / 'main.py').exists(), f'not the repo root: {ROOT_FOLDER}'  # Asserting GripVT root

root_str = str(ROOT_FOLDER)
if root_str not in sys.path:
    sys.path.insert(0, root_str)

RANDOM_SEED = 42
EXPERIMENT_NAME = "gripvt-baseline-models"          # For MLflow


## Section 1 — Load Data

In [ ]:
RECORDINGS = ROOT_FOLDER / "recordings"

csv_paths = sorted(RECORDINGS.glob("participant*/session*/features_table.csv"))

sessions = []
for p in csv_paths:
    folder_name = p.parent.name
    session_id = int(folder_name.removeprefix('session').split('_')[0])
    sessions.append((session_id, p))

assert len(sessions) == 9, f"expected 9 sessions, got {len(sessions)}"
assert sorted(s_id for s_id, _ in sessions) == [1, 2, 3, 5, 6, 7, 8, 9, 10]

In [ ]:
df_list = []
calibration_modes = set()
git_commits = set()

for session_id, p in sessions:
    feature_table = pd.read_csv(p)                                  # Reads features_table.csv for each path in sessions\

    participant_id = str(p.parent.parent.name.split('_')[0])        # Gets participant number
    feature_table['participant_id'] = participant_id
    feature_table['session_id'] = session_id

    df_list.append(feature_table)

    with open(p.parent / 'features_meta.json', encoding='utf-8') as f:
        meta = json.load(f)                                         # something to remember: load for file, loads for string

    calibration_modes.add(meta['calibration_mode'])

    git_commits.add(meta['git_commit'])

df = pd.concat(df_list, ignore_index=True)

assert df.shape == (28260, 808), f"expected (28260, 808), got {df.shape}"
assert df.index.is_unique, f"expected True, got {df.index.is_unique}"
assert df.index[-1] == 28259, f"expected True, got {df.index[-1]}"
assert pd.api.types.is_integer_dtype(df.session_id), f"expected int,  got {df.session_id.dtype}"
assert calibration_modes == {"per_wear"},  f"mixed calibration: {calibration_modes}"
assert git_commits == {"0018b43"}, f"mixed git commits: {git_commits}"

df.groupby("session_id").size()

## Section 2 — Read Folds

In [ ]:
with open(ROOT_FOLDER / 'notebooks' / 'split_manifest.json', encoding='utf-8') as f:
    manifest = json.load(f)

session_folds = manifest['session_folds']
loaded_sessions = set(df['session_id'].unique())

assert len(session_folds) == 9, f"len(session_folds) expected 9, got {len(session_folds)}"

for fold in session_folds:
    train_sessions = fold['train_sessions']
    test_session = fold['test_session']

    assert {test_session} | set(train_sessions) == loaded_sessions, \
        f"fold {fold['fold']}: sessions {sorted({test_session} | set(train_sessions))} != loaded {sorted(loaded_sessions)}"
    assert test_session not in train_sessions, \
        f"fold {fold['fold']}: test session {test_session} leaks into train_sessions"
    assert len(train_sessions) == 8, \
        f"fold {fold['fold']}: len(train_sessions) expected 8, got {len(train_sessions)}"

test_sessions = sorted(fold['test_session'] for fold in session_folds)
assert test_sessions == sorted(loaded_sessions), \
    f"each session must be held out exactly once, got {test_sessions}"

assert manifest['random_seed'] == RANDOM_SEED, \
    f"expected manifest random_seed {RANDOM_SEED}, got {manifest['random_seed']}"

## Section 3 — Dropping velocity-NaN rows

In [ ]:
df.isna().any(axis=1).sum()             # Checking how many rows have NaN values

In [ ]:
df = df.dropna(subset=df.filter(regex='vx|vy').columns)

assert df.shape[0] == 27991, f"expected row count is 27,991, got {df.shape[0]}"
assert df.isna().any().sum() == 0, f"NaN rows: {df[df.isna().any(axis=1)]}"

# A 1-frame trial would vanish entirely in the drop above, shrinking a fold silently.
# A trial is the pair (session_id, trial_id) -- trial_id alone runs 0-29 in every session.
n_trials = df.groupby(['session_id', 'trial_id']).ngroups
assert n_trials == 269, f"expected 269 trials after the drop, got {n_trials}"
assert df.shape[1] == 808, f"expected 808 columns, got {df.shape[1]}"

## Section 4 — Feature Matrix and Channel Groups

In [ ]:
# Getting unique columns prefixes (i.e. vx, vy, etc) 
unique_prefix = df.columns.str.replace(r'[\d_]', '', regex=True).unique()
print(unique_prefix)

# Remove non_feature rows
unwanted_prefixes = unique_prefix[~unique_prefix.isin(['dx', 'dy', 'dA', 'mag', 'strainx', 
                                                       'strainy', 'vx', 'vy', 'maxdisp'])].tolist()
print(unwanted_prefixes)

In [ ]:
unique_prefix = df.columns.str.replace(r'\d+$', '', regex=True).unique()        # Remove only the trailing numbers from the column names using r'\d+$'

print(f"Extracted Unique Prefixes: {unique_prefix}")

allowed_prefixes = ['dx_', 'dy_', 'dA_', 'mag_', 'strain_x_', 'strain_y_', 'vx_', 'vy_', 'max_disp']

unwanted_prefixes = unique_prefix[~pd.Index(unique_prefix).isin(allowed_prefixes)].tolist() # Filter out the allowed prefixes to find the unwanted ones

print(f"Unwanted Prefixes Identified: {unwanted_prefixes}\nCount: {len(unwanted_prefixes)}")

In [ ]:
# Store all feature columns in an ordered list
allowed_tuple = tuple(allowed_prefixes)         # Convert to tuple, required for str.startswith()
matching_cols = df.columns[df.columns.str.startswith(allowed_tuple)].tolist()

feature_cols = matching_cols
assert len(feature_cols) == 793, f"expected 793, got {len(feature_cols)}" 

TARGET = 'pct_mvc_corr'
assert TARGET not in feature_cols, f"pct_mvc_corr in feature_cols"

print(feature_cols)

In [ ]:
channel_groups = {
    "dx_dy": [c for c in feature_cols if 'dx' in c or 'dy' in c],
    "dA": [c for c in feature_cols if 'dA' in c],
    "mag": [c for c in feature_cols if 'mag' in c],
    "strain": [c for c in feature_cols if 'strain_x' in c or 'strain_y' in c],
    "velocity": [c for c in feature_cols if 'vx' in c or 'vy' in c],
    "max_disp": ['max_disp']
}

grouped_cols = [c for cols in channel_groups.values() for c in cols]

assert len(channel_groups) == 6, \
    f"expected 6 groups, got {len(channel_groups)}: {list(channel_groups.keys())}"

# Disjoint: a column claimed by two groups shows up twice in grouped_cols.
assert len(grouped_cols) == len(set(grouped_cols)), \
    f"groups overlap: {sorted({c for c in grouped_cols if grouped_cols.count(c) > 1})}"

# Covering: nothing left out, nothing invented. Disjoint + covering = a partition.
assert set(grouped_cols) == set(feature_cols), \
    f"missing: {sorted(set(feature_cols) - set(grouped_cols))}, extra: {sorted(set(grouped_cols) - set(feature_cols))}"

assert len(grouped_cols) == 793, f"expected 793 grouped columns, got {len(grouped_cols)}"

print({name: len(cols) for name, cols in channel_groups.items()})

## Section 5 — Metrics

One function used by every model, rung and ablation variant. Returns 9 rows: overall,
3 phase slices, 5 level slices. `braced` is a subset of `pre_cue`, so the phase slices
overlap — only the level slices partition the rows.

RMSE in %MVC points is the headline (directly comparable to EchoForce's 10.10 %).
R² is scored against each slice's own variance, so it goes sharply negative on narrow
slices even when RMSE is fine. That is arithmetic, not a broken model.

**Predict-the-mean baseline over all 27,991 rows — the floor any model must beat:**

| slice | n | mean_y | RMSE | MAE | R² |
|---|---|---|---|---|---|
| all | 27,991 | 7.40 | **15.03** | 9.86 | 0.000 |
| pre_cue | 13,436 | 1.32 | 7.12 | 6.68 | −2.669 |
| post_cue | 14,555 | 13.00 | 19.69 | 12.79 | −0.088 |
| braced | 4,365 | 1.16 | 6.33 | 6.24 | −33.677 |
| level 0 | 4,548 | 1.27 | 7.24 | 6.90 | −2.537 |
| level 5 | 5,247 | 2.08 | 5.70 | 5.34 | −6.693 |
| level 20 | 5,979 | 6.03 | 8.64 | 7.56 | −0.026 |
| level 50 | 5,939 | 11.32 | 17.55 | 12.65 | −0.053 |
| level 80 | 6,278 | 13.88 | 24.09 | 15.32 | −0.078 |

R² is exactly 0.000 on the `all` row by definition — that is the check that proves the
function is wired correctly. Level 80 averages only 13.88 %MVC, consistent with the
participant001 finding that 80 % MVC was never actually reached.

In [ ]:
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score

def evaluate(y_true, y_pred, meta):
    ''' One row per reporting slice: Overall, By Phase, By Level
        y_true  : 1-D array or Series of true %MVC
        y_pred  : 1-D numpy array, same length and same order
        meta    : DataFrame with 'phase', 'braced', 'target_pct', same length and order

        'braced' is a subset of 'pre_cue', so the phase slices overlap and their n do
        not sum to len(y_true). The five level slices do partition the rows.
    '''
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    # Masks go to numpy so every lookup below is positional, never index-aligned.
    slices = [("all", "all", np.ones(len(y_true), dtype=bool))]
    slices += [("phase", p, (meta["phase"] == p).to_numpy()) for p in ("pre_cue", "post_cue")]
    slices += [("phase", "braced", meta["braced"].to_numpy())]
    slices += [("level", lvl, (meta["target_pct"] == lvl).to_numpy()) for lvl in (0, 5, 20, 50, 80)]

    rows = []
    for slice_type, slice_name, mask in slices:
        yt, yp = y_true[mask], y_pred[mask]
        rows.append({
            "slice_type": slice_type,
            "slice": slice_name,
            "n": len(yt),
            "mean_y": np.mean(yt),
            "rmse": root_mean_squared_error(yt, yp),
            "mae": mean_absolute_error(yt, yp),
            "r2": r2_score(yt, yp),
        })

    return pd.DataFrame(rows)

In [ ]:
# Sanity-check evaluate() before any model exists.
meta_all = df[["phase", "braced", "target_pct"]]
y_all = df[TARGET]

perfect = evaluate(y_all, y_all.to_numpy(), meta_all)                       # predict the truth
constant = evaluate(y_all, np.full(len(y_all), y_all.mean()), meta_all)     # predict the mean

assert len(perfect) == 9, f"expected 9 slices, got {len(perfect)}"
assert (perfect["rmse"] == 0).all(), "perfect prediction must give RMSE 0 everywhere"
assert (perfect["r2"] == 1).all(), "perfect prediction must give R2 1 everywhere"
assert (constant["rmse"] >= constant["mae"]).all(), "RMSE < MAE is impossible"

# pre_cue + post_cue covers every row; braced is a subset of pre_cue and is left out.
phase_n = perfect.loc[perfect["slice"].isin(["pre_cue", "post_cue"]), "n"].sum()
level_n = perfect.loc[perfect["slice_type"] == "level", "n"].sum()
assert phase_n == len(df), f"phase slices cover {phase_n}, expected {len(df)}"
assert level_n == len(df), f"level slices cover {level_n}, expected {len(df)}"

constant.round(3)

## 6 — Nested Tuning Loop

In [ ]:
"""
    Take one fold dictionary -> pull out train_sessions and test_session
    ASSERT: test_session not in train_session
    Loop 8 times.  On loop i, one of the 8 training sessions becomes the inner validation session, and the other 7 become the inner training set.
    Each time: build fresh model from params, fit on 7, predict the 1, compute RMSE
    Return mean of 8 RMSE numbers, float
"""

def inner_cv_score(fold, params, make_model):
    ''' Mean RMSE over one outer fold's 8 inner leave-one-session-out splits.
        fold        : ONE dict from session_folds
        params      : hyperparameter dict, handed straight to make_model
        make_model  : callable(params) -> a fresh object with .fit / .predict

        The outer test session never appears here: inner splits are drawn only from this
        fold's 8 training sessions. Returns one float so Optuna has something to minimise.
        Plain aggregate RMSE, not evaluate() -- tuning needs one number, not 9 rows.
    '''
    train_sessions = fold['train_sessions']
    assert fold['test_session'] not in train_sessions, \
        f"fold {fold['fold']}: test session {fold['test_session']} leaks into {train_sessions}"

    scores = []
    for val_session in train_sessions:
        inner_train = [s for s in train_sessions if s != val_session]    # derive the 7, never mutate the 8

        train_mask = df['session_id'].isin(inner_train)                  # masks rebuilt every pass
        val_mask = df['session_id'] == val_session

        model = make_model(params)                                       # fresh model per split, no warm start
        model.fit(df.loc[train_mask, feature_cols], df.loc[train_mask, TARGET])
        y_pred = model.predict(df.loc[val_mask, feature_cols])

        scores.append(root_mean_squared_error(df.loc[val_mask, TARGET], y_pred))

    return float(np.mean(scores))

In [ ]:
# Smoke test: no Optuna yet. One fold, one fixed alpha, 8 inner fits -> one number and a timing.
# StandardScaler lives inside a Pipeline so it is fitted on the 7 training sessions only,
# never on all of df. That is the defence against the scaler leak.
import time
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

t0 = time.perf_counter()
smoke_score = inner_cv_score(
    session_folds[0],
    {},
    lambda p: Pipeline([("scale", StandardScaler()), ("ridge", Ridge(alpha=1.0))]),
)
print(f"fold 0 (test session {session_folds[0]['test_session']}) mean inner RMSE: {smoke_score:.3f} %MVC")
print(f"{time.perf_counter() - t0:.1f} s for 8 fits")

In [ ]:
# Two checks on inner_cv_score, both instant.
fold0 = session_folds[0]

# 1. The 8 training sessions hold 24,354 rows: 27,991 minus session 1's 3,637.
assert df['session_id'].isin(fold0['train_sessions']).sum() == 24354

# 2. The leak guard must actually fire. make_model is None because the assert raises
#    before any model is built. Copy the fold -- never mutate the manifest's list.
bad_fold = {**fold0, 'train_sessions': fold0['train_sessions'] + [fold0['test_session']]}
guard_fired = False
try:
    inner_cv_score(bad_fold, {}, None)
except AssertionError:
    guard_fired = True
assert guard_fired, "leak guard did not fire on a fold with the test session in train_sessions"

print("inner_cv_score: 24,354 training rows, leak guard fires")

In [ ]:
import time

# mlflow 3.15 REFUSES the legacy ./mlruns file store -- it raises "maintenance mode" unless
# you opt out with an env var. Point it at sqlite instead, which is what mlflow recommends.
# Read the results with:  mlflow ui --backend-store-uri sqlite:///mlflow.db
MLFLOW_DB = f"sqlite:///{(ROOT_FOLDER / 'mlflow.db').as_posix()}"
mlflow.set_tracking_uri(MLFLOW_DB)
mlflow.set_experiment(EXPERIMENT_NAME)

# Budget -- the SAME MAX_TRIALS and PATIENCE for every model. That is the fairness contract.
MAX_TRIALS = 100
PATIENCE = 15

optuna.logging.set_verbosity(optuna.logging.WARNING)


def stop_when_flat(study, trial):
    ''' Study-level early stopping -- the budget rule. Optuna ships no patience stopper, but
        the study already records which trial holds the best value, so "trials since the last
        improvement" needs no counter. Pruning stays OFF: this is the only stopping rule.
    '''
    if trial.number - study.best_trial.number >= PATIENCE:
        study.stop()

## Section 7 — Models

One cell per model, each holding its own flat 9-fold loop. The three cells are deliberate
copy-paste siblings: same folds, same `MAX_TRIALS` / `PATIENCE`, same `inner_cv_score`, same
`evaluate()`. Only the params dict and the model constructor differ — that is the fairness
contract, and it is what makes "XGBoost beat Ridge" a statement about models rather than
about scoring code.

### 7a - Ridge Regression

In [ ]:
# 7a -- Ridge.
# StandardScaler must live INSIDE the Pipeline: the Pipeline fits it on exactly the rows handed
# to .fit(), so it never sees the inner validation session or the outer test session.
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


def make_ridge(params):
    return Pipeline([("scale", StandardScaler()), ("ridge", Ridge(**params))])  # Data is normalized here

ridge_rows = []

# 1. Search, using 8 training sessions only
for fold in session_folds:
    def objective(trial):
        params = {
            "alpha": trial.suggest_float("alpha", 1e-3, 1e6, log=True)
        }
        return inner_cv_score(fold, params, make_ridge)

    # Run Optuna
    study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED))
    study.optimize(objective, n_trials=MAX_TRIALS, callbacks=[stop_when_flat])   # FIXED n_trial -> n_trials

    # 2. Refit on all 8, then touch the test session exactly once
    # FIXED: these were copied from inner_cv_score, so they referenced inner_train and
    # val_session -- names that only exist inside that function. Out here the split is the
    # OUTER one: all 8 training sessions vs the single held-out session.
    train_mask = df['session_id'].isin(fold['train_sessions'])
    test_mask = df['session_id'] == fold['test_session']

    model = make_ridge(study.best_params)
    t0 = time.perf_counter()
    model.fit(df.loc[train_mask, feature_cols], df.loc[train_mask, TARGET])
    fit_seconds = time.perf_counter() - t0

    y_pred = model.predict(df.loc[test_mask, feature_cols])
    y_true = df.loc[test_mask, TARGET]
    table = evaluate(y_true, y_pred, df.loc[test_mask, ['phase', 'braced', 'target_pct']])

    # 3. Label 9 rows so they survive concat
    table['model'] = 'ridge'
    table['fold'] = fold['fold']
    table['test_session'] = fold['test_session']
    table['n_trials'] = len(study.trials)
    table['fit_seconds'] = fit_seconds

    # 4. One MLflow run per (model, fold)
    # FIXED: log_metrics takes ONE dict of numbers, not a name and a value. The 'all' row also
    # carries the string columns slice_type/slice, which MLflow rejects, so name the numbers.
    overall = table.loc[table['slice'] == 'all'].iloc[0]

    with mlflow.start_run(run_name=f"ridge-fold{fold['fold']}"):
        mlflow.log_params(study.best_params)
        mlflow.log_metrics({
            "test_rmse": overall['rmse'],
            "test_mae": overall['mae'],
            "test_r2": overall['r2'],
            "inner_rmse_best": study.best_value,
            "n_trials": len(study.trials),
            "fit_seconds": fit_seconds,
        })

    # FIXED: print the real numbers -- nbstripout erases outputs, so this is the only live
    # feedback during the run, and the achieved trial count is itself a reported result.
    print(f"fold {fold['fold']} test{fold['test_session']}: RMSE {overall['rmse']:.3f} | "
          f"inner {study.best_value:.3f} | {len(study.trials)} trials | "
          f"alpha {study.best_params['alpha']:.4g}")

    ridge_rows.append(table)

ridge_results = pd.concat(ridge_rows, ignore_index=True)

In [ ]:
ridge_results.loc[ridge_results['slice'] == 'all', 'n'].sum()          # must be 27991

In [ ]:
# by-slice .mean() of n/rmse/mae/r2
ridge_results.groupby(['slice_type','slice'], sort=False)[['n','rmse','mae','r2']].mean().round(3)

In [ ]:
# per-fold all rows
ridge_results.loc[ridge_results['slice'] == 'all', ['test_session','rmse','mae','r2','n_trials']].round(3)

In [ ]:
# by-slice rmse.agg(['mean','std','min','max']) — the spread
ridge_results.groupby(['slice_type','slice'], sort=False)['rmse'].agg(['mean','std','min','max']).round(3)

In [ ]:
# Ridge, fold 8. `study` is 7a's leftover loop variable -- it holds the LAST fold of the Ridge
# run (count 47 confirms it), and the 7b loop below rebinds the same name to XGBoost.
# Capture it under a real name, guarded: re-running this cell after 7b must not silently
# swap an XGBoost study in behind a Ridge label.
if 'ridge_fold8_study' not in globals():
    ridge_fold8_study = study

ridge_fold8_study.trials_dataframe()['value'].describe()

### 7a results — Ridge, rung 1 (cross-mount, cold)

Run 2026-08-11 · 44 min · `MAX_TRIALS = 100`, `PATIENCE = 15`, seed 42.

Nine folds. Each one trains on eight sessions and is tested on the ninth, so every fold
answers: *how well does this work on a wearing of the sleeve it has never seen?*

#### Per fold

| fold | test session | test RMSE | inner RMSE | trials | alpha |
|---|---|---|---|---|---|
| 0 | 1 | 6.105 | 6.808 | 45 | 2.446e+04 |
| 1 | 2 | 7.460 | 6.752 | 97 | 3.137e+04 |
| 2 | 3 | 5.259 | 7.084 | **100 (cap)** | 4.384e+04 |
| 3 | 5 | 5.974 | 6.908 | 38 | 4.175e+04 |
| 4 | 6 | 7.318 | 6.743 | 38 | 4.175e+04 |
| 5 | 7 | **13.216** | 6.069 | 31 | 2.485e+04 |
| 6 | 8 | 8.219 | 6.901 | 47 | 5.136e+04 |
| 7 | 9 | 4.619 | 7.206 | 47 | 5.136e+04 |
| 8 | 10 | 4.897 | 7.202 | 47 | 5.136e+04 |

**Mean 7.007 %MVC, SD 2.632, range 4.62 – 13.22.** Predicting the training mean scores 15.03,
so Ridge is well clear of the floor.

Two things to say carefully:

- This is the **average of nine per-fold RMSEs**. Pooling all 27,991 predictions into one RMSE
  instead gives **7.295**. The average-of-folds is the one to quote — it is what SO4 asks for
  and it is the same quantity as EchoForce's 10.10 % — but say which one you mean.
- **Session 7 (13.216) is not excluded.** It is 9.7 % of the rows but 31.9 % of the total
  squared error. Dropping it would be picking the data after seeing the score. Without it the
  mean is 6.231 (SD 1.312), pooled 6.336. Report both; exclude neither. The session that *was*
  excluded is **session 4**, on an a-priori rule — its force export contains zero data rows, so
  it has no labels to train against. Its marker data is fine: a ground-truth acquisition
  failure, not a sensor failure.

#### What the phase slices mean

Every trial has a moment where the operator says "go" — the `grip_start` event. `to_cue` is the
number of seconds from that moment: negative before, positive after. The three slices are just
windows on that clock.

| slice | when | what the participant is doing |
|---|---|---|
| `pre_cue` | before "go" | waiting, at rest — around 1 %MVC no matter what the trial is labelled |
| `post_cue` | after "go" | actually squeezing |
| `braced` | the last 1 s before "go" | tensing up in anticipation. This is **part of `pre_cue`**, not a fourth phase |

Because `braced` sits inside `pre_cue`, the three row counts do not add up to the total. Only
the five target levels partition the rows.

#### By phase and by level

RMSE columns are across the nine folds: mean, standard deviation, and the best and worst mount.
The spread columns come from cell 30 (`groupby(...)['rmse'].agg(['mean','std','min','max'])`),
recorded here because `nbstripout` erases cell outputs on commit.

| slice | rows | RMSE | **SD** | **min** | **max** | MAE | R² | floor |
|---|---|---|---|---|---|---|---|---|
| **all** | 27,991 | **7.007** | 2.632 | 4.619 | 13.216 | 4.492 | 0.755 | 15.03 |
| `pre_cue` | 13,436 | 4.048 | 1.637 | 2.299 | 7.134 | 2.922 | −0.661 | 7.12 |
| **`post_cue`** | 14,555 | **8.764** | **3.165** | **6.001** | **16.147** | 5.874 | 0.752 | 19.69 |
| `braced` | 4,365 | 4.070 | 1.837 | 2.130 | 7.893 | 3.150 | −21.399 | 6.33 |
| level 0 | 4,548 | 3.590 | 1.791 | 2.001 | 7.151 | 2.716 | −54.015 | 7.24 |
| level 5 | 5,247 | 3.158 | 1.751 | 1.393 | 6.001 | 2.640 | −1.903 | 5.70 |
| level 20 | 5,979 | 5.831 | 2.549 | 3.702 | 11.521 | 4.329 | 0.436 | 8.64 |
| level 50 | 5,939 | 7.868 | 3.255 | 4.851 | 15.221 | 5.448 | 0.752 | 17.55 |
| level 80 | 6,278 | **10.347** | **4.135** | 5.934 | 19.565 | 6.652 | 0.778 | 24.09 |

**The number to quote is post-cue 8.764, not the aggregate 7.007.**

About half of every row in the dataset is pre-cue rest sitting near zero force. Predicting near
zero when the truth is near zero is easy, and the model does it well (4.048). So the aggregate
is an average of an easy interval and a hard one — it credits the model for the half of the
recording where there is nothing to measure. The sensor's job is to read force *while someone
is gripping*, and that is the post-cue interval alone. Post-cue 8.764 is still under EchoForce's
10.10 %.

The same effect shows up in the level labels: the "80 %" trials average only 13.9 %MVC across
their full duration, because half of each one is rest. **The levels are instructed targets, not
achieved force.**

#### The spread is the SO4 deliverable

**Post-cue ranges 6.001 – 16.147 across the nine mounts, SD 3.165.** That is the sentence SO4
asks for — *how much accuracy varies from one wearing to the next* — and it is wider than the
aggregate row suggests (SD 2.632). EchoForce quotes a 6.01 – 14.34 % range beside its 10.10 %
headline, which makes this the like-for-like comparison: **GripVT's post-cue range is comparable
in width and lower in centre.**

Level 80 is the worst case: SD 4.135, range 5.934 – 19.565 — a threefold swing between the best
and worst mount on the hardest target.

Three notes on reading the table:

- `braced` (4.070) is essentially identical to `pre_cue` (4.048), so anticipatory tension does
  not appear to confuse the model.
- **Negative R² on `braced` and level 0 is arithmetic, not failure.** R² compares the model
  against that slice's own variance, and those slices are nearly constant, so the denominator
  collapses. RMSE is the headline for exactly this reason. These are also averages of nine
  per-fold R², which is unbounded below — one bad fold can drag level 0 to −54.
- Every slice's row count matches the §5 floor table exactly, which proves each row was
  predicted **exactly once** across the nine folds. No fold dropped or double-counted rows.

#### Cost, and one caveat about the search

490 trials total, mean 54 per fold, **1 of 9 hit the 100-trial cap.** About 0.7 s per Ridge fit,
roughly 5,100 fits.

⚠ **The repeated alphas are not nine independent searches agreeing.** Folds 6–8 report the same
alpha *and* the same trial count, as do folds 3–4. `TPESampler` draws its first ten trials from
the seeded RNG, so every fold explores the same first ten alphas before the search adapts
(verified: two studies with unrelated objectives gave identical trials 0–9). The defensible
claim is *"the tuned penalty was consistently large, 2.4–5.1×10⁴"* — nothing stronger.

The objective is also nearly flat near its optimum. Fold 8's 47 trials ranged 7.202 – 9.670,
with a quarter of them within 0.083 of the best. That is why the patience rule often fails to
fire and studies ride the cap — it counts any improvement at all, including 1e-9.

### 7b - XGBoost

A copy-paste sibling of 7a: same folds, same `MAX_TRIALS` / `PATIENCE`, same `inner_cv_score`,
same `evaluate()`. Only the params dict and the constructor differ. **No `Pipeline` and no
scaler** — trees split on thresholds, so feature scale is irrelevant and there is no fitted
transform that could leak.

#### D1 — capacity: `n_estimators` is tuned as an ordinary knob

Protocol §3 says capacity should be **data-determined by training early stopping, never tuned**.
That rule is not honoured here, and the deviation is deliberate:

- Training early stopping needs a watch set. `inner_cv_score` calls `model.fit(X, y)` and has
  none. Supplying one means a uniform `fit(X, y, X_val, y_val)` signature across all three
  models — wrapper classes, the abstraction the flat-notebook revision deliberately removed —
  and it would still leave the **final refit** without a watch set.
- It buys **one knob out of seven.** XGBoost carries `learning_rate`, `max_depth`, `subsample`,
  `colsample_bytree`, `min_child_weight` and `reg_lambda` against Ridge's single `alpha` either
  way, so the effort imbalance §3 exists to close stays open regardless.

The Methods sentence is therefore *"capacity was tuned alongside the other hyperparameters under
the same budget."*

⚠ **The cost of this, stated rather than hidden:** a fold that hits the 100-trial cap has
under-searched a 7-dimensional space, where Ridge hit the same cap having under-searched a
1-dimensional one. Same budget, very different coverage. `n_trials` is recorded per fold so the
reader can see it. Raising the cap for XGBoost alone would void the comparison, so it stays at
100 — see protocol §10.

#### D5 — the search space

Four of the seven knobs are **runtime decisions as much as modelling ones**, because they set
how much data each tree touches:

| knob | range | cost effect |
|---|---|---|
| `n_estimators` | 50 – 300, log | ~linear |
| `colsample_bytree` | 0.3 – 0.8 | linear in columns scanned (0.3 → 238 of 793 features) |
| `subsample` | 0.5 – 1.0 | linear in rows scanned |
| `max_depth` | 3 – 8 | ~linear |
| `learning_rate` | 0.02 – 0.3, log | none |
| `min_child_weight` | 1 – 20 | none |
| `reg_lambda` | 1e-2 – 1e3, log | none |

`log=True` on `n_estimators` is doing runtime work as well as modelling work: it puts about half
the draws below ~120 trees without forbidding 300.

Projected against the one measured point (300 trees, all 793 columns, 24,354 rows → 9.69 s on
the RTX 4060): **~4 h if all nine folds ride the 100-trial cap**, ~2.5 h at Ridge's 54-trial
average. Budget for the cap — a 7-dimensional flat objective will find microscopic improvements
far more often than Ridge's 1-dimensional one did, and `stop_when_flat` counts any improvement
including 1e-9. The probe cell below replaces this arithmetic with measurement.

`device="cuda"` and `tree_method="hist"` are **fixed hardware settings, not searched.** Dropping
`device="cuda"` falls back to CPU at ~3.5× the cost (35 s/fit) and raises nothing — the run
simply does not finish overnight.

In [ ]:
from xgboost import XGBRegressor

fold = session_folds[0]
train_mask = df['session_id'].isin(fold['train_sessions'])

xgb = XGBRegressor(device="cuda", tree_method="hist", n_estimators=300, random_state=RANDOM_SEED)

t0 = time.perf_counter()
xgb.fit(df.loc[train_mask, feature_cols], df.loc[train_mask, TARGET])
print(f"{time.perf_counter() - t0:.2f} s for one fit on {train_mask.sum():,} rows x {len(feature_cols)} features")

In [ ]:
# Pre-launch probe -- replaces the runtime projection with measurement. ~1 min. Run it BEFORE
# the 7b loop below and read the last column: that is the worst case, all 9 folds at the cap.
#
# Timed on an INNER-sized split (7 sessions, not 8), because that is the fit inner_cv_score
# performs up to 7,200 times. The cell above measured an outer-sized fit, which is ~14% larger.
#
# The predict is timed too. It is the first place a CUDA-trained booster handed a pandas
# DataFrame can emit the device-mismatch warning. WATCH HOW MANY TIMES IT PRINTS: once means
# Python is de-duplicating it and the overnight run stays readable; three times means it is not,
# and 7,200 fits would bury the per-fold prints. Say so and we add a warnings filter.

probe_fold = session_folds[0]
probe_val = probe_fold['train_sessions'][0]
probe_train = [s for s in probe_fold['train_sessions'] if s != probe_val]

probe_train_mask = df['session_id'].isin(probe_train)
probe_val_mask = df['session_id'] == probe_val

# Three corners of the D5 space. "median" is roughly where a random draw lands.
corners = {
    "cheap":     dict(n_estimators=50,  max_depth=3, subsample=0.5,  colsample_bytree=0.3),
    "median":    dict(n_estimators=120, max_depth=6, subsample=0.75, colsample_bytree=0.55),
    "expensive": dict(n_estimators=300, max_depth=8, subsample=1.0,  colsample_bytree=0.8),
}

print(f"probe: fit on {probe_train_mask.sum():,} rows (7 sessions) x {len(feature_cols)} features, "
      f"predict {probe_val_mask.sum():,}\n")

for name, p in corners.items():
    m = XGBRegressor(**p, learning_rate=0.1, device="cuda", tree_method="hist",
                     random_state=RANDOM_SEED)

    t0 = time.perf_counter()
    m.fit(df.loc[probe_train_mask, feature_cols], df.loc[probe_train_mask, TARGET])
    fit_s = time.perf_counter() - t0

    t0 = time.perf_counter()
    m.predict(df.loc[probe_val_mask, feature_cols])
    predict_s = time.perf_counter() - t0

    trial_s = 8 * (fit_s + predict_s)                    # one Optuna trial = 8 inner splits
    print(f"{name:>9}: {fit_s:6.2f} s fit + {predict_s:5.2f} s predict "
          f"-> {trial_s:6.1f} s/trial -> {900 * trial_s / 3600:5.2f} h if all 9 folds cap out")

In [ ]:
# 7b -- XGBoost. A copy of the 7a cell: same folds, same MAX_TRIALS / PATIENCE, same
# inner_cv_score, same evaluate(). Only the params dict and the constructor differ.
#
# No Pipeline and no scaler here. Trees split on thresholds, so feature scale is irrelevant
# and there is no fitted transform that could leak into a validation or test session.

def make_xgb(params):
    ''' device="cuda" and tree_method="hist" are FIXED hardware settings, never searched.
        Dropping device="cuda" silently falls back to CPU at ~3.5x the cost (35 s/fit) --
        no error is raised, the run simply does not finish overnight.
    '''
    return XGBRegressor(**params, device="cuda", tree_method="hist", random_state=RANDOM_SEED)


# Crash insurance. MLflow stores only the 'all' row per fold, so a kernel death at fold 8 would
# lose every by-phase and by-level row for folds 0-7. Append each fold's 9 rows to disk as they
# finish instead. Deleted first: mode='a' onto a leftover file would silently double the results.
XGB_LIVE_CSV = ROOT_FOLDER / 'notebooks' / 'xgb_results_live.csv'
XGB_LIVE_CSV.unlink(missing_ok=True)

xgb_rows = []
run_t0 = time.perf_counter()

# 1. Search, using 8 training sessions only
for fold in session_folds:
    def objective(trial):
        # D1 = A: n_estimators is an ordinary tuned knob, log-scaled so half the draws land
        # below ~120 trees. See the markdown above for the protocol section 3 deviation.
        params = {
            "n_estimators":     trial.suggest_int("n_estimators", 50, 300, log=True),
            "learning_rate":    trial.suggest_float("learning_rate", 0.02, 0.3, log=True),
            "max_depth":        trial.suggest_int("max_depth", 3, 8),
            "subsample":        trial.suggest_float("subsample", 0.5, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.3, 0.8),
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
            "reg_lambda":       trial.suggest_float("reg_lambda", 1e-2, 1e3, log=True),
        }
        return inner_cv_score(fold, params, make_xgb)

    fold_t0 = time.perf_counter()

    study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED))
    study.optimize(objective, n_trials=MAX_TRIALS, callbacks=[stop_when_flat])

    # 2. Refit on all 8, then touch the test session exactly once
    train_mask = df['session_id'].isin(fold['train_sessions'])
    test_mask = df['session_id'] == fold['test_session']

    model = make_xgb(study.best_params)
    t0 = time.perf_counter()
    model.fit(df.loc[train_mask, feature_cols], df.loc[train_mask, TARGET])
    fit_seconds = time.perf_counter() - t0

    y_pred = model.predict(df.loc[test_mask, feature_cols])
    y_true = df.loc[test_mask, TARGET]
    table = evaluate(y_true, y_pred, df.loc[test_mask, ['phase', 'braced', 'target_pct']])

    # 3. Label 9 rows so they survive concat
    table['model'] = 'xgboost'
    table['fold'] = fold['fold']
    table['test_session'] = fold['test_session']
    table['n_trials'] = len(study.trials)
    table['fit_seconds'] = fit_seconds

    # Written before MLflow so the disk copy exists even if the tracking call fails.
    table.to_csv(XGB_LIVE_CSV, mode='a', header=not XGB_LIVE_CSV.exists(), index=False)

    # 4. One MLflow run per (model, fold)
    overall = table.loc[table['slice'] == 'all'].iloc[0]

    with mlflow.start_run(run_name=f"xgboost-fold{fold['fold']}"):
        mlflow.log_params(study.best_params)
        mlflow.log_metrics({
            "test_rmse": overall['rmse'],
            "test_mae": overall['mae'],
            "test_r2": overall['r2'],
            "inner_rmse_best": study.best_value,
            "n_trials": len(study.trials),
            "fit_seconds": fit_seconds,
        })

    # nbstripout erases outputs, so this print is the only live feedback during the run.
    # The two elapsed columns are what tell you, at fold 2, whether the run lands by morning.
    bp = study.best_params
    print(f"fold {fold['fold']} test{fold['test_session']}: RMSE {overall['rmse']:.3f} | "
          f"inner {study.best_value:.3f} | {len(study.trials)} trials | "
          f"n_est {bp['n_estimators']} depth {bp['max_depth']} "
          f"lr {bp['learning_rate']:.3f} colsample {bp['colsample_bytree']:.2f} | "
          f"fold {(time.perf_counter() - fold_t0) / 60:.1f} min, "
          f"total {(time.perf_counter() - run_t0) / 60:.1f} min", flush=True)

    xgb_rows.append(table)

xgb_results = pd.concat(xgb_rows, ignore_index=True)
print(f"\n7b complete: {(time.perf_counter() - run_t0) / 60:.1f} min, "
      f"{xgb_results.loc[xgb_results['slice'] == 'all', 'n_trials'].sum()} trials total")

In [ ]:
xgb_results.loc[xgb_results['slice'] == 'all', 'n'].sum()          # must be 27991

In [ ]:
# by-slice .mean() of n/rmse/mae/r2
xgb_results.groupby(['slice_type','slice'], sort=False)[['n','rmse','mae','r2']].mean().round(3)

In [ ]:
# per-fold all rows
xgb_results.loc[xgb_results['slice'] == 'all', ['test_session','rmse','mae','r2','n_trials']].round(3)

In [ ]:
# by-slice rmse.agg(['mean','std','min','max']) — the spread. This is the SO4 deliverable.
xgb_results.groupby(['slice_type','slice'], sort=False)['rmse'].agg(['mean','std','min','max']).round(3)

In [ ]:
# XGBoost, fold 8's study -- how flat is the objective? Compare against Ridge fold 8
# (47 trials, min 7.202, std 0.924, a quarter within 0.083 of the best). `study` is the 7b
# loop's leftover variable, so this is fold 8 and nothing else.
study.trials_dataframe()['value'].describe()

### 7b results — XGBoost, rung 1 (cross-mount, cold)

Run 2026-08-11 · **164.2 min** · `MAX_TRIALS = 100`, `PATIENCE = 15`, seed 42, GPU (`device="cuda"`,
`tree_method="hist"`). Same nine folds, same budget and same `evaluate()` as 7a.

#### Per fold

| fold | test session | test RMSE | inner RMSE | trials | n_est | depth | lr | colsample |
|---|---|---|---|---|---|---|---|---|
| 0 | 1 | 7.006 | 6.319 | 33 | 158 | 8 | 0.152 | 0.58 |
| 1 | 2 | 6.946 | 6.624 | 40 | 119 | 7 | 0.077 | 0.74 |
| 2 | 3 | 4.381 | 6.848 | 38 | 161 | 7 | 0.056 | 0.45 |
| 3 | 5 | 5.935 | 6.813 | 17 | 236 | 7 | 0.102 | 0.78 |
| 4 | 6 | 7.685 | 6.442 | 45 | 298 | 5 | 0.057 | 0.35 |
| 5 | 7 | **10.567** | 6.151 | 72 | 204 | 7 | 0.075 | 0.46 |
| 6 | 8 | 8.615 | 6.389 | 41 | 114 | 7 | 0.056 | 0.72 |
| 7 | 9 | 4.159 | 6.874 | 37 | 231 | 5 | 0.156 | 0.44 |
| 8 | 10 | 4.251 | 6.975 | 47 | 262 | 7 | 0.026 | 0.75 |

**Mean 6.616 %MVC, SD 2.182, range 4.159 – 10.567.** Pooling all 27,991 predictions instead gives
**6.838**. Excluding session 7: mean **6.122** (SD 1.713), pooled 6.306. Floor is 15.03.

Row counts reproduce the §5 floor table exactly, so every row was predicted **exactly once**
across the nine folds.

#### By phase and by level, with Ridge alongside

| slice | rows | RMSE | SD | min | max | MAE | R² | **Ridge RMSE** | floor |
|---|---|---|---|---|---|---|---|---|---|
| **all** | 27,991 | **6.616** | 2.182 | 4.159 | 10.567 | 3.286 | 0.789 | 7.007 | 15.03 |
| `pre_cue` | 13,436 | 2.719 | 1.174 | 1.759 | 5.368 | 1.254 | 0.247 | 4.048 | 7.12 |
| **`post_cue`** | 14,555 | **8.679** | **2.669** | **5.497** | **13.394** | 5.084 | 0.767 | **8.764** | 19.69 |
| `braced` | 4,365 | 2.288 | 1.980 | 0.960 | 7.185 | 1.393 | −6.716 | 4.070 | 6.33 |
| level 0 | 4,548 | 1.872 | 1.278 | 0.485 | 3.873 | 0.965 | −5.709 | 3.590 | 7.24 |
| level 5 | 5,247 | 2.324 | 1.978 | 1.015 | 7.356 | 1.566 | −0.707 | 3.158 | 5.70 |
| level 20 | 5,979 | 5.650 | 2.158 | 2.911 | 9.144 | 3.332 | 0.490 | 5.831 | 8.64 |
| level 50 | 5,939 | 7.353 | 2.108 | 5.318 | 11.195 | 4.117 | 0.794 | 7.868 | 17.55 |
| level 80 | 6,278 | **9.976** | 4.298 | 5.353 | 17.906 | 5.648 | 0.792 | 10.347 | 24.09 |

---

### ⚠ The headline gap is not what it looks like

The aggregate says XGBoost beats Ridge by **0.391** (6.616 vs 7.007). Both of the cuts that
matter shrink that to nearly nothing.

**Cut 1 — by phase. Post-cue 8.679 vs 8.764 is a gap of 0.085 %MVC, about 1 %.** On the interval
the sensor exists to measure, the two models are indistinguishable. Almost all of XGBoost's
aggregate advantage comes from the **pre-cue rest** interval (2.719 vs 4.048, a 1.33-point gain)
and its subset `braced` (2.288 vs 4.070). XGBoost is much better at predicting *near-zero force
while nothing is happening* — which is the easy half of the recording, and roughly 48 % of the
rows. The level slices improve for the same reason: each level trial is about half rest, so
level 0 (1.872 vs 3.590) and level 5 (2.324 vs 3.158) are largely the same rest effect
reappearing.

**Cut 2 — per fold. Six of nine mounts improve, three get worse, and one session carries the
average.**

| test session | Ridge | XGBoost | Δ |
|---|---|---|---|
| 1 | 6.105 | 7.006 | **+0.901** |
| 2 | 7.460 | 6.946 | −0.514 |
| 3 | 5.259 | 4.381 | −0.878 |
| 5 | 5.974 | 5.935 | −0.039 |
| 6 | 7.318 | 7.685 | **+0.367** |
| **7** | **13.216** | **10.567** | **−2.649** |
| 8 | 8.219 | 8.615 | **+0.396** |
| 9 | 4.619 | 4.159 | −0.460 |
| 10 | 4.897 | 4.251 | −0.646 |
| **mean** | 7.007 | 6.616 | **−0.391** |
| **mean excl. session 7** | 6.231 | 6.122 | **−0.109** |

**Remove session 7 and the two models differ by 0.109 %MVC across eight mounts** — well inside
the fold-to-fold noise, which has an SD above 1.7 for both. The defensible sentence is
*"gradient boosting did not improve on the linear model during the grip interval"*, not
*"XGBoost outperformed Ridge."*

Session 7 stays in — it passed the a-priori screen, and dropping it on a model score is post-hoc
selection. It is 9.7 % of the rows and **23.2 %** of the total squared error, down from Ridge's
31.9 %.

### Where XGBoost does earn something: the spread

This is the SO4-relevant result, and it is about **repeatability rather than accuracy**.

| | Ridge | XGBoost |
|---|---|---|
| all — SD / range | 2.632 / 4.619 – 13.216 | **2.182 / 4.159 – 10.567** |
| **post_cue — SD / range** | 3.165 / 6.001 – 16.147 | **2.669 / 5.497 – 13.394** |
| level 80 — SD / range | 4.135 / 5.934 – 19.565 | 4.298 / 5.353 – 17.906 |

The worst mount improves by **2.6 points** overall and **2.8 points** post-cue, and the spread
narrows on both. EchoForce quotes 6.01 – 14.34 % beside its 10.10 % headline; GripVT's post-cue
range of 5.497 – 13.394 is comparable in width and lower at both ends. So the honest claim is
that boosting bought **consistency across wearings**, not per-mount accuracy — which is closer to
what SO4 asks about than the mean is.

Level 80 is the one place the spread does **not** improve (SD 4.298 vs 4.135), even though its
mean does. The hardest target stays the least repeatable.

### Cost, and what it says about the D1 deviation

**370 trials, mean 41 per fold, range 17 – 72, and _zero_ folds hit the 100-trial cap.** Ridge
used 490 trials and hit the cap once.

That inverts the worry recorded in the 7b header. The concern was that a 7-dimensional space
under a budget sized for a 1-dimensional one would ride the cap; instead the patience rule fired
*earlier* for XGBoost than for Ridge, so **the larger space received the smaller search**. Same
rule, applied identically, both models converged under it — which is exactly what the budget
rule was written to do, but the direction is worth stating rather than glossing.

Fold 8's objective is also less flat than Ridge's: SD **0.406** against Ridge's 0.924, best 6.975
with a quarter of trials within 0.116. Adding knobs made the surface easier for patience to call,
not harder.

⚠ **The seed caveat that applied to Ridge does not apply here.** All nine folds still draw the
same first ten trials (`TPESampler`, `n_startup_trials = 10`, seeded), but with seven dimensions
no two folds selected the same configuration — `n_estimators` ranges 114 – 298 and no pair
repeats. Unlike Ridge's repeated alphas, these are nine genuinely different outcomes.

**On the search space:** TPE selected `max_depth` 7 or 8 on **seven of nine folds**. Capping the
range at 6 to protect the overnight window — considered before launch, from the probe's 42.7 h
worst-case projection — would have excluded the region the search actually preferred. The full
3 – 8 range ran in 2.74 h.

### Cost comparison (protocol §4, fourth cut)

| | Ridge | XGBoost |
|---|---|---|
| wall clock | 44 min | **164 min** |
| trials (total / mean / cap hits) | 490 / 54 / 1 | 370 / 41 / **0** |
| model fits | ~5,100 | ~3,000 |
| per fit | ~0.7 s | ~2.5 s (GPU) |

XGBoost costs **3.7× the wall clock** for a post-cue gain of 0.085 %MVC. Cost is part of the
evaluation, not a footnote — and on this dataset the tree model does not pay for itself on
accuracy. Its case rests on the narrower spread.

### Still to note

- **Inner and test RMSE correlate negatively again, r = −0.933** (−0.900 excluding session 7),
  the same figure Ridge produced (−0.95 / −0.81). This is **arithmetic, not a finding**: one
  intrinsically hard mount makes the inner score look good precisely when that mount is the one
  held out. Do not present it as a discovery.
- **Negative R² on `braced` and level 0 is still arithmetic**, but both improved markedly
  (−6.716 vs −21.399; −5.709 vs −54.015), and `pre_cue` R² turned positive (0.247 vs −0.661).
  RMSE remains the headline.
- The CUDA device-mismatch warning at `.predict()` appeared **once** — XGBoost self-limits it.
  It cost roughly 0.15 s per predict, about 18 min over the whole run.
- Raw per-fold rows were written to `notebooks/xgb_results_live.csv` as each fold finished
  (gitignored, derived). Per-fold `all` metrics are in `mlflow.db` under `xgboost-fold*`.